# Avance 02 — C5 · Etapa 3: aplicación de reglas de limpieza
**Equipo A&C | Proyecto de Inteligencia de Negocios**

En esta etapa conservamos los 1,341,419 registros delimitados y validamos campos derivados sin alterar la fuente. En la Etapa 2 identificamos **10 folios repetidos (20 filas)**, **0 duplicados exactos en las 17 columnas originales**, **76,204 discrepancias de alcaldía** y **1,544 horas de creación faltantes**. No confundimos un folio repetido con una fila duplicada. Nuestra prioridad es preservar evidencia y evitar pérdidas injustificadas.

**Unidad de observación:** registro de incidente C5. No sumamos directamente estos registros con los hechos de tránsito SSC.

## 1. Lectura reproducible
Buscamos el archivo delimitado por la Etapa 1 y trabajamos por bloques para evitar sobrecargar la memoria de JupyterLab. Si hay varias copias, pedimos una ruta explícita.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from collections import Counter
from IPython.display import display

RUTA_ETAPA1_MANUAL = None  # Opcional: Path('/ruta/resultados/resultados_C5/etapa1')
TAM_BLOQUE = 50000
NOMBRE_ENTRADA = 'incidentes_viales_C5_etapa1.csv'
if RUTA_ETAPA1_MANUAL is not None:
    ENTRADA = Path(RUTA_ETAPA1_MANUAL).expanduser().resolve()
else:
    cwd = Path.cwd().resolve()
    opciones = []
    for base in [cwd, *cwd.parents]:
        opciones += [base/'resultados'/'resultados_C5'/'etapa1', base/'resultados_C5'/'etapa1']
    opciones = [p for p in opciones if (p/NOMBRE_ENTRADA).is_file()]
    if not opciones:
        opciones = [p for p in Path.home().rglob('etapa1') if p.parent.name == 'resultados_C5' and (p/NOMBRE_ENTRADA).is_file()]
    opciones = sorted(set(p.resolve() for p in opciones))
    if len(opciones) != 1:
        print('Carpetas encontradas:', *opciones, sep='\n - ')
        raise RuntimeError('Selecciona la ruta exacta en RUTA_ETAPA1_MANUAL.')
    ENTRADA = opciones[0]
ARCHIVO = ENTRADA/NOMBRE_ENTRADA
SALIDA = ENTRADA.parent/'etapa3'
SALIDA.mkdir(parents=True, exist_ok=True)
print('Entrada:', ARCHIVO)
print('Salida:', SALIDA)

Entrada: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_C5/etapa1/incidentes_viales_C5_etapa1.csv
Salida: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_C5/etapa3


## 2. Contrato de columnas y reglas
Mantenemos todas las columnas originales. Creamos `hora_creacion_validada` y `hora_cierre_validada` sin inventar valores, `requiere_revision_alcaldia`, `estado_calidad_alcaldia`, `folio_repetido` y `requiere_revision_folio`. Las columnas originales permanecen intactas.

- **Alcaldía:** si las columnas normalizadas discrepan, conservamos `alcaldia_normalizada` de Etapa 1, pero la marcamos como **provisional**; no resolvemos el conflicto por mayoría.
- **Folios:** conservamos las 20 filas con folios repetidos. Guardamos una evidencia completa de estos casos.
- **Horas:** valores ausentes o fuera de `HH:MM[:SS]` quedan nulos solo en la columna derivada.
- **Duplicados:** no eliminamos filas; la Etapa 2 detectó cero duplicados exactos en 17 campos.
- **Coordenadas:** el rango global válido no demuestra pertenencia a CDMX; queda pendiente la comprobación espacial.
- **Fechas:** preservamos fechas de creación y cierre. No imputamos fechas.

In [2]:
ORIGINALES = ['folio','fecha_creacion','hora_creacion','dia_semana','fecha_cierre',
    'hora_cierre','tipo_incidente_c4','incidente_c4','alcaldia_inicio',
    'codigo_cierre','clas_con_f_alarma','tipo_entrada','alcaldia_cierre',
    'alcaldia_catalogo','colonia_catalogo','longitud','latitud']
COLUMNAS_REQUERIDAS = ORIGINALES + ['archivo_origen','fila_origen',
    'alcaldia_catalogo_norm','alcaldia_inicio_norm','alcaldia_cierre_norm',
    'alcaldia_normalizada','alcaldias_discrepantes']
columnas_entrada = pd.read_csv(ARCHIVO, nrows=0, encoding='utf-8-sig').columns.tolist()
faltan = [c for c in COLUMNAS_REQUERIDAS if c not in columnas_entrada]
assert not faltan, 'Faltan columnas: ' + str(faltan)
print('Columnas originales en CSV delimitado:', len(columnas_entrada))

Columnas originales en CSV delimitado: 26


## 3. Identificación de folios repetidos
Contamos folios utilizables por bloques, ignorando marcadores como `SD`. En una segunda lectura marcaremos los folios repetidos sin eliminar registros.

In [3]:
SIN_FOLIO = {'','SD','S/D','ND','N/D','NA','N/A','SIN DATO','SIN DATOS'}
conteo_folios = Counter()
filas_total = 0
for bloque in pd.read_csv(ARCHIVO, dtype=str, chunksize=TAM_BLOQUE, encoding='utf-8-sig', low_memory=False, usecols=['folio']):
    fol = bloque['folio'].fillna('').str.strip().str.upper()
    conteo_folios.update(fol[~fol.isin(SIN_FOLIO)].value_counts().to_dict())
    filas_total += len(bloque)
folios_repetidos = {folio for folio, veces in conteo_folios.items() if veces > 1}
resumen_folios = pd.DataFrame([{'filas_entrada':filas_total,
    'folios_distintos_utilizables':len(conteo_folios),
    'folios_utilizables_repetidos':len(folios_repetidos),
    'filas_en_folios_repetidos':sum(conteo_folios[f] for f in folios_repetidos),
    'maximo_filas_por_folio':max(conteo_folios.values()) if conteo_folios else 0}])
display(resumen_folios)
resumen_folios.to_csv(SALIDA/'resumen_folios_etapa3_C5.csv',index=False,encoding='utf-8-sig')

,filas_entrada,folios_distintos_utilizables,folios_utilizables_repetidos,filas_en_folios_repetidos,maximo_filas_por_folio
0,1341419,1341409,10,20,2


## 4. Transformación no destructiva y exportación
Procesamos por bloques, añadimos indicadores de calidad y exportamos un CSV completo. Los 20 registros de folios repetidos y hasta 200 ejemplos de discrepancias geográficas se guardan por separado. La columna de alcaldía de Etapa 1 no se reemplaza.

In [4]:
def validar_hora(serie):
    t = serie.astype('string').str.strip()
    p = t.str.extract(r'^(\d{1,2}):(\d{2})(?::(\d{2}))?$')
    h = pd.to_numeric(p[0],errors='coerce')
    m = pd.to_numeric(p[1],errors='coerce')
    s = pd.to_numeric(p[2],errors='coerce').fillna(0)
    ok = h.between(0,23) & m.between(0,59) & s.between(0,59)
    return t.where(ok, pd.NA), ok.fillna(False)

ARCHIVO_SALIDA = SALIDA/'incidentes_viales_C5_etapa3.csv'
ARCHIVO_FOLIOS = SALIDA/'folios_repetidos_casos_C5.csv'
ARCHIVO_DISCREPANCIAS = SALIDA/'muestra_alcaldias_discrepantes_C5.csv'
for ruta in [ARCHIVO_SALIDA, ARCHIVO_FOLIOS, ARCHIVO_DISCREPANCIAS]:
    if ruta.exists(): ruta.unlink()  # Reejecución reproducible: solo se reemplazan salidas de Etapa 3

conteos = Counter()
muestra_guardada = 0
for i, bloque in enumerate(pd.read_csv(ARCHIVO, dtype=str, chunksize=TAM_BLOQUE,
                                      encoding='utf-8-sig', low_memory=False), start=1):
    original = bloque.copy(deep=True)
    fol = bloque['folio'].fillna('').str.strip().str.upper()
    bloque['folio_repetido'] = fol.isin(folios_repetidos)
    bloque['requiere_revision_folio'] = bloque['folio_repetido']
    for col in ['hora_creacion','hora_cierre']:
        validada, ok = validar_hora(bloque[col])
        bloque[col+'_validada'] = validada
        bloque['estado_'+col] = np.select(
            [bloque[col].isna() | bloque[col].str.strip().eq(''), ~ok],
            ['faltante','formato_invalido'], default='valida')
        conteos[col+'_faltante'] += int((bloque['estado_'+col]=='faltante').sum())
        conteos[col+'_invalida'] += int((bloque['estado_'+col]=='formato_invalido').sum())
    columnas_alc = ['alcaldia_catalogo_norm','alcaldia_inicio_norm','alcaldia_cierre_norm']
    # Solo comparamos alcaldías informativas; las ausencias se registran por separado.
    valores = bloque[columnas_alc].replace('',pd.NA)
    discrepante = valores.apply(lambda f: f.dropna().nunique()>1, axis=1)
    bloque['requiere_revision_alcaldia'] = discrepante
    bloque['estado_calidad_alcaldia'] = np.where(discrepante,'discrepante_revisar','sin_discrepancia_detectada')
    assert bloque[columnas_entrada].equals(original[columnas_entrada]), 'Se alteraron columnas de entrada.'
    conteos['filas_exportadas'] += len(bloque)
    conteos['folios_repetidos_filas'] += int(bloque['folio_repetido'].sum())
    conteos['alcaldias_discrepantes'] += int(discrepante.sum())
    bloque.to_csv(ARCHIVO_SALIDA, mode='a', index=False, header=i==1, encoding='utf-8-sig' if i==1 else 'utf-8')
    casos = bloque.loc[bloque['folio_repetido']]
    if len(casos):
        casos.to_csv(ARCHIVO_FOLIOS, mode='a', index=False, header=not ARCHIVO_FOLIOS.exists(),encoding='utf-8-sig' if not ARCHIVO_FOLIOS.exists() else 'utf-8')
    restante = 200 - muestra_guardada
    if restante > 0:
        casos_alc = bloque.loc[discrepante].head(restante)
        if len(casos_alc):
            casos_alc.to_csv(ARCHIVO_DISCREPANCIAS,mode='a',index=False,
                header=not ARCHIVO_DISCREPANCIAS.exists(),
                encoding='utf-8-sig' if not ARCHIVO_DISCREPANCIAS.exists() else 'utf-8')
            muestra_guardada += len(casos_alc)
    if i%5==0: print('Bloques:',i,'| Filas exportadas:',conteos['filas_exportadas'])
print('CSV creado:', ARCHIVO_SALIDA)

Bloques: 5 | Filas exportadas: 250000
Bloques: 10 | Filas exportadas: 500000
Bloques: 15 | Filas exportadas: 750000
Bloques: 20 | Filas exportadas: 1000000
Bloques: 25 | Filas exportadas: 1250000
CSV creado: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados/resultados_C5/etapa3/incidentes_viales_C5_etapa3.csv


## 5. Control de impacto y comprobaciones
Comparamos los conteos con la Etapa 2. Si un resultado no coincide, no interpretamos el archivo como definitivo. **No hemos ejecutado una corrección automática de alcaldías ni eliminado duplicados.**

In [5]:
impacto = pd.DataFrame([{
    'filas_recibidas':filas_total,
    'filas_exportadas':conteos['filas_exportadas'],
    'filas_eliminadas':filas_total-conteos['filas_exportadas'],
    'folios_repetidos_filas':conteos['folios_repetidos_filas'],
    'alcaldias_discrepantes':conteos['alcaldias_discrepantes'],
    'hora_creacion_faltante':conteos['hora_creacion_faltante'],
    'hora_cierre_faltante':conteos['hora_cierre_faltante'],
    'columnas_entrada':len(columnas_entrada)
}])
display(impacto)
impacto.to_csv(SALIDA/'impacto_etapa3_C5.csv',index=False,encoding='utf-8-sig')
comprobaciones = {
    'mismas_filas_que_etapa1': filas_total == 1341419,
    'sin_filas_eliminadas': conteos['filas_exportadas'] == filas_total,
    'folios_repetidos_esperados': len(folios_repetidos)==10 and conteos['folios_repetidos_filas']==20,
    'discrepancias_esperadas': conteos['alcaldias_discrepantes']==76204,
    'horas_creacion_faltantes_esperadas': conteos['hora_creacion_faltante']==1544,
    'horas_cierre_faltantes_esperadas': conteos['hora_cierre_faltante']==1544,
    'archivo_salida_existe': ARCHIVO_SALIDA.is_file(),
    'entrada_sigue_disponible': ARCHIVO.is_file()
}
comprobaciones_df = pd.DataFrame([{'comprobacion':k,'cumple':v} for k,v in comprobaciones.items()])
display(comprobaciones_df)
comprobaciones_df.to_csv(SALIDA/'comprobaciones_etapa3_C5.csv',index=False,encoding='utf-8-sig')
assert all(comprobaciones.values()), 'Alguna cifra no coincide; revisar las reglas antes de cerrar.'
print('Etapa 3: comprobaciones superadas.')

,filas_recibidas,filas_exportadas,filas_eliminadas,folios_repetidos_filas,alcaldias_discrepantes,hora_creacion_faltante,hora_cierre_faltante,columnas_entrada
0,1341419,1341419,0,20,76204,1544,1544,26


,comprobacion,cumple
0,mismas_filas_que_etapa1,True
1,sin_filas_eliminadas,True
2,folios_repetidos_esperados,True
3,discrepancias_esperadas,True
4,horas_creacion_faltantes_esperadas,True
5,horas_cierre_faltantes_esperadas,True
6,archivo_salida_existe,True
7,entrada_sigue_disponible,True


Etapa 3: comprobaciones superadas.


## 6. Bitácora de decisiones y conclusión
En esta etapa **conservamos los 1,341,419 registros** y añadimos columnas derivadas para identificar horas no válidas, folios repetidos y alcaldías discrepantes. **No eliminamos** las 20 filas asociadas a los 10 folios repetidos, porque no son duplicados exactos según el análisis previo de 17 columnas. Tampoco resolvimos arbitrariamente las 76,204 discrepancias de alcaldía.

**Pendientes:** revisar las 20 filas de folios repetidos y sus diferencias; contrastar alcaldías con coordenadas y criterios espaciales documentados; comprobar la cobertura de 2024; definir cómo se relacionarán C5 y SSC sin duplicar incidentes. Los resultados de C5 son complementarios, no necesariamente equivalentes a los de SSC.